# Notebook 6 - Prepare Model-Ready Datasets

## Airbnb Capstone | Feature preparation before modelling

**Inputs:** `madrid_master_model_dataset.csv` and `tokyo_master_model_dataset.csv`

This notebook creates slimmer model-ready datasets from the richer master datasets. It keeps the master data untouched, removes modelling clutter, and adds a small engineered feature layer before training city-specific models.

### Structure
1. Setup and helper functions
2. Prepare model-ready datasets
3. Review audit output
4. Inspect dropped and engineered features
5. Validate final files


---
## 1. Optional Colab Setup

In [ ]:
# Optional Colab setup
# from google.colab import drive
# drive.mount('/content/drive')
# %cd /content/drive/MyDrive/CAPSTONE

Run this only in Google Colab if the notebook cannot already see the uploaded CAPSTONE folder. Locally, leave it commented out.

---
## 2. Setup, Imports, and Project Paths

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd


def get_project_root() -> Path:
    """Find the CAPSTONE folder from any notebook subfolder."""
    cwd = Path.cwd().resolve()
    candidates = [cwd, *cwd.parents]
    for candidate in candidates:
        data_dir = candidate / "1. Data"
        if (data_dir / "master" / "airbnb_clean_final.csv").exists() or (data_dir / "raw" / "madrid_listings.csv").exists():
            return candidate
    raise FileNotFoundError("Could not find project root containing the 1. Data folder. Check the notebook working directory.")


PROJECT_ROOT = get_project_root()
MASTER_DIR = PROJECT_ROOT / "1. Data" / "master"
OUTPUT_DIR = PROJECT_ROOT / "1. Data" / "model_ready"

TARGET_COLUMNS = ["price_eur", "log_price_eur"]

# Explicitly excluded from model-ready CSVs.

This cell imports the required libraries, finds the CAPSTONE project folder, and defines the master-data input folder, model-ready output folder, and target columns.

---
## 3. Drop Rules and Feature Dictionary

In [ ]:
DROP_COLUMNS = {
    # identifiers/context that should not predict city-specific prices
    "listing_id",
    "city",
    # raw dates; useful for audit, not directly model-friendly
    "last_scraped",
    "host_since",
    "first_review",
    "last_review",
    # redundant or non-informative for first pass
    "property_type",  # keep property_group instead
    "host_total_listings_count",  # keep host_listings_count + local calculated counts
    "minimum_nights",  # keep capped version
    "has_availability",  # constant or near-constant
    "calendar_days",  # essentially fixed annual calendar length
    "calendar_available_days",  # redundant with unavailable days/rate
    "calendar_unavailable_days",  # redundant with unavailable rate
    "raw_review_total_reviews",  # equal to number_of_reviews when matched
    "raw_reviews_last_6m",  # keep cleaned reviews_last_6m instead
}

These are fields intentionally excluded from the model-ready files, such as identifiers, raw dates, duplicated fields, and variables kept mainly for audit or chatbot context.

In [ ]:
FEATURE_DICTIONARY = [
    ("price_eur", "target", "Nightly listing price in EUR."),
    ("log_price_eur", "target_transform", "Log-transformed target for skewed price modelling."),
    ("neighbourhood_cleansed", "location", "Main local market effect."),
    ("neighbourhood_group", "location", "Broader neighbourhood grouping where available."),
    ("latitude", "location", "Fine-grained geographic signal."),
    ("longitude", "location", "Fine-grained geographic signal."),
    ("distance_to_center_km", "location_engineered", "Centrality proxy."),
    ("is_central_5km", "location_engineered", "Flag for listings within 5 km of city center."),
    ("neighbourhood_listing_count", "location_engineered", "Local Airbnb market density."),
    ("property_group", "property", "Simplified property category."),
    ("room_type", "property", "Major price driver: entire home/private/shared/hotel room."),
    ("accommodates", "property", "Capacity signal."),
    ("bedrooms", "property", "Size signal."),
    ("beds", "property", "Size signal."),
    ("bathrooms", "property", "Comfort/size signal."),
    ("bathroom_type", "property", "Shared/private/standard bathroom setup."),
    ("beds_per_guest", "property_engineered", "Capacity comfort ratio."),
    ("bathrooms_per_guest", "property_engineered", "Comfort/crowding ratio."),
    ("capacity_bucket", "property_engineered", "Coarse guest-capacity segment."),
    ("amenities_count", "amenities", "Overall amenity richness."),
    ("has_wifi", "amenities", "Convenience/value signal."),
    ("has_kitchen", "amenities", "Long-stay/family value signal."),
    ("has_air_conditioning", "amenities", "Comfort signal."),
    ("has_washer", "amenities", "Long-stay convenience signal."),
    ("has_dedicated_workspace", "amenities", "Remote-work/business signal."),
    ("has_tv", "amenities", "Comfort signal."),
    ("has_parking", "amenities", "Convenience signal."),
    ("has_elevator", "amenities", "Accessibility/convenience signal."),
    ("has_heating", "amenities", "Comfort signal."),
    ("has_self_checkin", "amenities", "Convenience/operational signal."),
    ("host_response_time", "host", "Host responsiveness category."),
    ("host_response_rate", "host", "Host responsiveness rate."),
    ("host_acceptance_rate", "host", "Booking acceptance signal."),
    ("host_is_superhost", "host", "Trust/reputation signal."),
    ("host_listings_count", "host", "Host scale signal."),
    ("host_has_profile_pic", "host", "Trust/completeness signal."),
    ("host_identity_verified", "host", "Trust signal."),
    ("calculated_host_listings_count", "host", "Local host portfolio size."),
    ("calculated_host_listings_count_entire_homes", "host", "Host portfolio composition."),
    ("calculated_host_listings_count_private_rooms", "host", "Host portfolio composition."),
    ("calculated_host_listings_count_shared_rooms", "host", "Host portfolio composition."),
    ("host_tenure_days", "host_engineered", "Host experience signal."),
    ("is_professional_host", "host_engineered", "Multi-listing/professional host flag."),
    ("host_scale_bucket", "host_engineered", "Single/small/professional/large host segment."),
    ("instant_bookable", "booking", "Booking friction/convenience signal."),
    ("minimum_nights_capped", "booking", "Stay-rule feature with extreme values capped."),
    ("maximum_nights", "booking", "Stay-rule feature."),
    ("availability_30", "availability", "Near-term availability."),
    ("availability_60", "availability", "Short-term availability."),
    ("availability_90", "availability", "Medium-term availability."),
    ("availability_365", "availability", "Annual availability."),
    ("low_availability_30", "availability_engineered", "Near-term scarcity/demand pressure flag."),
    ("high_annual_availability", "availability_engineered", "High annual availability flag."),
    ("availability_30_ratio", "availability_engineered", "Normalized 30-day availability."),
    ("availability_365_ratio", "availability_engineered", "Normalized annual availability."),
    ("calendar_unavailable_rate", "calendar", "Listing-level unavailable-rate proxy."),
    ("calendar_unavailable_q1", "calendar", "Quarterly seasonality proxy."),
    ("calendar_unavailable_q2", "calendar", "Quarterly seasonality proxy."),
    ("calendar_unavailable_q3", "calendar", "Quarterly seasonality proxy."),
    ("calendar_unavailable_q4", "calendar", "Quarterly seasonality proxy."),
    ("calendar_unavailable_autumn", "calendar", "Seasonality proxy."),
    ("calendar_unavailable_spring", "calendar", "Seasonality proxy."),
    ("calendar_unavailable_summer", "calendar", "Seasonality proxy."),
    ("calendar_unavailable_winter", "calendar", "Seasonality proxy."),
    ("calendar_unavailable_weekday", "calendar", "Weekday unavailable-rate proxy."),
    ("calendar_unavailable_weekend", "calendar", "Weekend unavailable-rate proxy."),
    ("calendar_weekend_unavailable_gap", "calendar_engineered", "Weekend minus weekday unavailable-rate gap."),
    ("number_of_reviews", "reviews", "Total review volume."),
    ("number_of_reviews_ltm", "reviews", "Review activity in the last 12 months."),
    ("number_of_reviews_l30d", "reviews", "Very recent review activity."),
    ("number_of_reviews_ly", "reviews", "Prior-year review activity."),
    ("reviews_last_6m", "reviews_engineered", "Six-month review momentum from corrected raw reviews."),
    ("reviews_per_month", "reviews", "Normalized review velocity."),
    ("has_reviews", "reviews", "Flag for listings with reviews."),
    ("has_rating", "reviews_engineered", "Flag for listings with rating scores."),
    ("days_since_last_review", "reviews", "Review recency."),
    ("review_history_days", "reviews", "Length of listing review history."),
    ("review_recency_bucket", "reviews_engineered", "No/recent/stale review recency segment."),
    ("reviewed_last_30d", "reviews_engineered", "Very recent review flag."),
    ("reviewed_last_90d", "reviews_engineered", "Recent review flag."),
    ("review_scores_rating", "ratings", "Overall quality score."),
    ("review_scores_accuracy", "ratings", "Sub-rating score."),
    ("review_scores_cleanliness", "ratings", "Sub-rating score."),
    ("review_scores_checkin", "ratings", "Sub-rating score."),
    ("review_scores_communication", "ratings", "Sub-rating score."),
    ("review_scores_location", "ratings", "Guest-perceived location score."),
    ("review_scores_value", "ratings", "Guest-perceived value score."),
]

This dictionary documents the retained and engineered modelling attributes so the modelling dataset remains explainable.

---
## 4. Engineered Feature Helpers

In [ ]:
def safe_ratio(numerator: pd.Series, denominator: pd.Series) -> pd.Series:
    denominator = denominator.replace(0, np.nan)
    return (numerator / denominator).replace([np.inf, -np.inf], np.nan)

This helper calculates ratios while avoiding divide-by-zero issues.

In [ ]:
def capacity_bucket(accommodates: pd.Series) -> pd.Series:
    return pd.cut(
        accommodates,
        bins=[0, 1, 2, 4, 6, np.inf],
        labels=["solo", "couple", "small_group", "family_group", "large_group"],
        include_lowest=True,
    ).astype("object")

This groups guest capacity into interpretable small, medium, and large listing buckets.

In [ ]:
def host_scale_bucket(host_listings_count: pd.Series) -> pd.Series:
    return pd.cut(
        host_listings_count.fillna(0),
        bins=[-1, 1, 4, 20, np.inf],
        labels=["single_listing", "small_multi_host", "professional_host", "large_operator"],
    ).astype("object")

This converts host listing counts into host-scale categories.

In [ ]:
def review_recency_bucket(df: pd.DataFrame) -> pd.Series:
    days = df["days_since_last_review"]
    conditions = [
        df["has_reviews"].fillna(0).eq(0) | days.isna(),
        days <= 30,
        days <= 90,
        days <= 365,
    ]
    choices = ["no_reviews", "reviewed_30d", "reviewed_90d", "reviewed_365d"]
    return pd.Series(
        np.select(conditions, choices, default="stale_over_365d"),
        index=df.index,
    )

This groups review recency into interpretable buckets for listings with recent, older, or missing review activity.

In [ ]:
def add_engineered_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    df["beds_per_guest"] = safe_ratio(df["beds"], df["accommodates"]).round(4)
    df["bathrooms_per_guest"] = safe_ratio(df["bathrooms"], df["accommodates"]).round(4)
    df["capacity_bucket"] = capacity_bucket(df["accommodates"])

    df["host_scale_bucket"] = host_scale_bucket(df["host_listings_count"])

    df["reviews_last_6m"] = df["raw_reviews_last_6m"].fillna(0)
    df["has_rating"] = df["review_scores_rating"].notna().astype(int)
    df["review_recency_bucket"] = review_recency_bucket(df)
    df["reviewed_last_30d"] = (
        df["has_reviews"].fillna(0).eq(1) & df["days_since_last_review"].le(30)
    ).astype(int)
    df["reviewed_last_90d"] = (
        df["has_reviews"].fillna(0).eq(1) & df["days_since_last_review"].le(90)
    ).astype(int)

    df["low_availability_30"] = df["availability_30"].le(7).astype(int)
    df["high_annual_availability"] = df["availability_365"].ge(270).astype(int)
    df["availability_30_ratio"] = safe_ratio(df["availability_30"], pd.Series(30, index=df.index)).round(4)
    df["availability_365_ratio"] = safe_ratio(df["availability_365"], pd.Series(365, index=df.index)).round(4)

    df["calendar_weekend_unavailable_gap"] = (
        df["calendar_unavailable_weekend"] - df["calendar_unavailable_weekday"]
    ).round(4)

    return df

This function adds the modelling features derived from existing columns, including capacity ratios, review recency, host scale, price-per-guest, and availability pressure flags.

---
## 5. Drop Sparse, Constant, and Explicitly Excluded Fields

In [ ]:
def drop_sparse_and_constant_features(df: pd.DataFrame, city: str) -> tuple[pd.DataFrame, pd.DataFrame]:
    rows = []
    keep_cols = []

    for col in df.columns:
        missing_pct = df[col].isna().mean() * 100
        unique_values = df[col].nunique(dropna=True)

        if col in TARGET_COLUMNS:
            decision = "keep_target"
            reason = "Target or target transform."
            keep = True
        elif col in DROP_COLUMNS:
            decision = "drop"
            reason = "Identifier, raw date, redundant, constant, or not model-friendly for first pass."
            keep = False
        elif missing_pct > 35:
            decision = "drop"
            reason = "More than 35% missing in this city."
            keep = False
        elif unique_values <= 1:
            decision = "drop"
            reason = "No useful variation within this city."
            keep = False
        else:
            decision = "keep_feature"
            reason = "Usable first-pass predictor."
            keep = True

        if keep:
            keep_cols.append(col)

        rows.append(
            {
                "city": city,
                "attribute": col,
                "decision": decision,
                "missing_pct": round(missing_pct, 3),
                "unique_values": int(unique_values),
                "reason": reason,
            }
        )

    return df[keep_cols].copy(), pd.DataFrame(rows)

This function applies the drop rules, removes constant or overly sparse attributes, keeps the target columns, and records a decision table explaining every keep/drop choice.

---
## 6. Order Columns and Prepare Each City

In [ ]:
def ordered_columns(df: pd.DataFrame) -> list[str]:
    target = [col for col in TARGET_COLUMNS if col in df.columns]
    categorical_priority = [
        "neighbourhood_cleansed",
        "neighbourhood_group",
        "property_group",
        "room_type",
        "bathroom_type",
        "capacity_bucket",
        "host_response_time",
        "host_scale_bucket",
        "review_recency_bucket",
    ]
    categorical = [col for col in categorical_priority if col in df.columns]
    remaining = [col for col in df.columns if col not in target + categorical]
    return target + categorical + remaining

This keeps the target columns first, followed by categorical fields and then the remaining model predictors.

In [ ]:
def prepare_city(city_key: str, file_name: str) -> tuple[pd.DataFrame, pd.DataFrame, dict[str, object]]:
    source = MASTER_DIR / file_name
    df = pd.read_csv(source)
    original_shape = df.shape
    df = add_engineered_features(df)
    engineered_cols = set(df.columns) - set(pd.read_csv(source, nrows=0).columns)
    model_ready, decisions = drop_sparse_and_constant_features(df, city_key.title())
    model_ready = model_ready[ordered_columns(model_ready)]

    audit = {
        "city": city_key.title(),
        "source_file": source.name,
        "source_rows": original_shape[0],
        "source_columns": original_shape[1],
        "model_ready_rows": model_ready.shape[0],
        "model_ready_columns": model_ready.shape[1],
        "feature_columns": model_ready.shape[1] - len(TARGET_COLUMNS),
        "dropped_columns": int((decisions["decision"] == "drop").sum()),
        "engineered_features_added": len(engineered_cols),
        "target_median_eur": round(float(model_ready["price_eur"].median()), 4),
        "target_mean_eur": round(float(model_ready["price_eur"].mean()), 4),
    }
    return model_ready, decisions, audit

This function loads one city master dataset, adds engineered features, applies the drop decisions, orders the columns, and returns the model-ready data plus audit information.

---
## 7. Write Outputs

In [ ]:
def ensure_output_dir() -> None:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

This creates the model-ready output folder if it does not already exist.

In [ ]:
def write_feature_dictionary() -> None:
    dictionary = pd.DataFrame(
        FEATURE_DICTIONARY,
        columns=["attribute", "feature_group", "modelling_value"],
    )
    dictionary.to_csv(OUTPUT_DIR / "model_ready_feature_dictionary.csv", index=False)

This writes the feature dictionary CSV used for reporting and later interpretation.

In [ ]:
def main() -> None:
    ensure_output_dir()
    city_files = {
        "madrid": "madrid_master_model_dataset.csv",
        "tokyo": "tokyo_master_model_dataset.csv",
    }

    audits = []
    all_decisions = []
    for city_key, file_name in city_files.items():
        print(f"\nPreparing model-ready dataset for {city_key.title()}...")
        model_ready, decisions, audit = prepare_city(city_key, file_name)
        output_file = OUTPUT_DIR / f"{city_key}_model_ready.csv"
        model_ready.to_csv(output_file, index=False)
        decisions.to_csv(OUTPUT_DIR / f"{city_key}_feature_decisions.csv", index=False)
        audits.append(audit)
        all_decisions.append(decisions)
        print(f"  saved: {output_file}")
        print(f"  shape: {model_ready.shape}")

    pd.DataFrame(audits).to_csv(OUTPUT_DIR / "model_ready_audit.csv", index=False)
    pd.concat(all_decisions, ignore_index=True).to_csv(
        OUTPUT_DIR / "model_ready_feature_decisions.csv",
        index=False,
    )
    write_feature_dictionary()

    print(f"\nModel-ready outputs saved to: {OUTPUT_DIR}")
    print(pd.DataFrame(audits).to_string(index=False))

This is the main runner. It prepares Madrid and Tokyo, saves the model-ready CSV files, writes audit/decision outputs, and prints the final audit summary.

---
## 8. Prepare Model-Ready Datasets

In [2]:
main()



Preparing model-ready dataset for Madrid...
  saved: C:\Users\georg\Desktop\Docs\Student Files\CAPSTONE\1. Data\Outputs\model_ready\madrid_model_ready.csv
  shape: (17770, 87)

Preparing model-ready dataset for Tokyo...
  saved: C:\Users\georg\Desktop\Docs\Student Files\CAPSTONE\1. Data\Outputs\model_ready\tokyo_model_ready.csv
  shape: (23765, 86)

Model-ready outputs saved to: C:\Users\georg\Desktop\Docs\Student Files\CAPSTONE\1. Data\Outputs\model_ready
  city                     source_file  source_rows  source_columns  model_ready_rows  model_ready_columns  feature_columns  dropped_columns  engineered_features_added  target_median_eur  target_mean_eur
Madrid madrid_master_model_dataset.csv        17770              88             17770                   87               85               15                         14            105.000         114.5985
 Tokyo  tokyo_master_model_dataset.csv        23765              88             23765                   86               84       

This creates `madrid_model_ready.csv` and `tokyo_model_ready.csv`. It drops identifiers, raw dates, redundant fields, and constant columns, then adds the curated engineered features.


---
## 9. Review Audit Output

In [3]:
audit = pd.read_csv(OUTPUT_DIR / "model_ready_audit.csv")
audit


,city,source_file,source_rows,source_columns,model_ready_rows,model_ready_columns,feature_columns,dropped_columns,engineered_features_added,target_median_eur,target_mean_eur
0,Madrid,madrid_master_model_dataset.csv,17770,88,17770,87,85,15,14,105.000,114.5985
1,Tokyo,tokyo_master_model_dataset.csv,23765,88,23765,86,84,16,14,95.914,109.8130


This audit confirms the number of rows, columns, dropped fields, engineered features, and target price statistics in the model-ready files.


---
## 10. Inspect Dropped and Engineered Features

In [4]:
decisions = pd.read_csv(OUTPUT_DIR / "model_ready_feature_decisions.csv")
decisions[decisions["decision"] == "drop"].sort_values(["city", "attribute"])


,city,attribute,decision,missing_pct,unique_values,reason
73,Madrid,calendar_available_days,drop,0.000,366,"Identifier, raw date, redundant, constant, or ..."
72,Madrid,calendar_days,drop,0.000,2,"Identifier, raw date, redundant, constant, or ..."
74,Madrid,calendar_unavailable_days,drop,0.000,366,"Identifier, raw date, redundant, constant, or ..."
1,Madrid,city,drop,0.000,1,"Identifier, raw date, redundant, constant, or ..."
51,Madrid,first_review,drop,15.898,3154,"Identifier, raw date, redundant, constant, or ..."
42,Madrid,has_availability,drop,0.574,1,"Identifier, raw date, redundant, constant, or ..."
31,Madrid,host_since,drop,0.495,3482,"Identifier, raw date, redundant, constant, or ..."
37,Madrid,host_total_listings_count,drop,0.495,139,"Identifier, raw date, redundant, constant, or ..."
52,Madrid,last_review,drop,15.898,1029,"Identifier, raw date, redundant, constant, or ..."
30,Madrid,last_scraped,drop,0.000,2,"Identifier, raw date, redundant, constant, or ..."


This table shows which attributes were dropped and why. The goal is to make the modelling dataset cleaner without losing the richer master dataset for the future chatbot.


In [5]:
feature_dictionary = pd.read_csv(OUTPUT_DIR / "model_ready_feature_dictionary.csv")
feature_dictionary[feature_dictionary["feature_group"].str.contains("engineered", na=False)]


,attribute,feature_group,modelling_value
6,distance_to_center_km,location_engineered,Centrality proxy.
7,is_central_5km,location_engineered,Flag for listings within 5 km of city center.
8,neighbourhood_listing_count,location_engineered,Local Airbnb market density.
16,beds_per_guest,property_engineered,Capacity comfort ratio.
17,bathrooms_per_guest,property_engineered,Comfort/crowding ratio.
18,capacity_bucket,property_engineered,Coarse guest-capacity segment.
41,host_tenure_days,host_engineered,Host experience signal.
42,is_professional_host,host_engineered,Multi-listing/professional host flag.
43,host_scale_bucket,host_engineered,Single/small/professional/large host segment.
51,low_availability_30,availability_engineered,Near-term scarcity/demand pressure flag.


This displays the engineered features added before modelling, such as capacity ratios, review recency buckets, host scale buckets, and availability pressure flags.


---
## 11. Validate Final Files

In [6]:
for city in ["madrid", "tokyo"]:
    path = OUTPUT_DIR / f"{city}_model_ready.csv"
    df = pd.read_csv(path)
    print(city.title(), df.shape)
    print("  target median:", round(df["price_eur"].median(), 2))
    print("  top missing columns:")
    print((df.isna().mean() * 100).sort_values(ascending=False).head(8).round(2))
    print()


Madrid (17770, 87)
  target median: 105.0
  top missing columns:
review_scores_cleanliness      15.9
review_scores_checkin          15.9
review_scores_communication    15.9
review_scores_value            15.9
review_scores_accuracy         15.9
review_scores_rating           15.9
review_scores_location         15.9
reviews_per_month              15.9
dtype: float64

Tokyo (23765, 86)
  target median: 95.91
  top missing columns:
review_scores_communication    13.06
review_scores_checkin          13.06
review_scores_cleanliness      13.06
review_scores_accuracy         13.06
review_scores_value            13.06
review_scores_location         13.06
reviews_per_month              13.06
review_scores_rating           13.06
dtype: float64



This final validation checks that both model-ready files load correctly and summarizes the remaining missingness before modelling.
